In [1]:
from pathlib import Path
import copy
import shutil
import numpy as np
import pandas as pd
import pycolmap
import torch
from scipy.spatial.transform import Rotation as R
from scipy.optimize import least_squares
import json
from datetime import datetime

from hloc import extract_features, match_features, pairs_from_retrieval, visualization
from hloc.localize_sfm import QueryLocalizer, pose_from_cluster
from hloc.utils.parsers import parse_retrieval
from hloc.utils import viz_3d

print("Gerekli kütüphaneler yüklendi")

Gerekli kütüphaneler yüklendi


## Adım 1: Ayarlar

In [2]:
# Bundle (harita) ve veri yolları
bundle_root = Path("../outputs/dragos-bundle")
sfm_model = bundle_root / "sparse"
images_dir = bundle_root / "images"  # reconstruction DB (train)

# Split edilmiş veri
train_images_dir = Path("../datasets/dragos/images/simulation/train")
test_images_dir = Path("../datasets/dragos/images/simulation/test")

# Tek GT dosyası
ground_truth_csv = Path("../datasets/dragos/images/simulation/simulation.csv")

# Çıktı klasörü
results_dir = bundle_root / "benchmark_results"
results_dir.mkdir(parents=True, exist_ok=True)

# Lokalizasyon parametreleri
num_loc = 10  # Retrieved DB image sayısı
max_error = 12  # RANSAC max error (pixels)

# Konfigürasyonlar
feature_conf = copy.deepcopy(extract_features.confs["superpoint_aachen"])
feature_conf["preprocessing"]["resize_max"] = 1024
retrieval_conf = extract_features.confs["netvlad"]
matcher_conf = match_features.confs["superpoint+lightglue"]

print(f"Bundle root: {bundle_root}")
print(f"Train images: {train_images_dir}")
print(f"Test images: {test_images_dir}")
print(f"Ground truth CSV: {ground_truth_csv}")
print(f"Sonuçlar: {results_dir}")

Bundle root: ..\outputs\dragos-bundle
Train images: ..\datasets\dragos\images\simulation\train
Test images: ..\datasets\dragos\images\simulation\test
Ground truth CSV: ..\datasets\dragos\images\simulation\simulation.csv
Sonuçlar: ..\outputs\dragos-bundle\benchmark_results


## Adım 2: Ground Truth Verilerini Yükle

In [3]:
def load_ground_truth(csv_path):
    """
    Tek simulation.csv dosyasından ground truth yükler.
    image adı birebir eşleştiği için key olarak sadece dosya adı kullanılır.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Ground truth CSV bulunamadı: {csv_path}")

    df = pd.read_csv(csv_path)
    gt_data = {}

    for _, row in df.iterrows():
        image_name = Path(row["image_path"]).name
        gt_data[image_name] = {
            "lat": row["gps_lat"],
            "lon": row["gps_lon"],
            "alt": row["gps_alt"],
            "heading_deg": row["heading_deg"],
            "timestamp": row["timestamp"],
            "image_path": row["image_path"],
        }

    print(f"✓ Ground truth yüklendi: {len(gt_data)} satır")
    return gt_data


gt_data = load_ground_truth(ground_truth_csv)

✓ Ground truth yüklendi: 133 satır


## Adım 3: GPS → Kartezyen Koordinatlar (Reference frame)

In [4]:
def gps_to_cartesian(lat, lon, alt, ref_lat, ref_lon, ref_alt):
    """
    GPS koordinatlarını reference noktasından Kartezyen koordinatlara dönüştür.
    Küçük mesafelerde (lokal alan) yeterli yaklaşım.
    """
    R_earth = 6.371e6  # metre

    lat_rad = np.radians(lat)
    lon_rad = np.radians(lon)
    ref_lat_rad = np.radians(ref_lat)
    ref_lon_rad = np.radians(ref_lon)

    dlat = lat_rad - ref_lat_rad
    dlon = lon_rad - ref_lon_rad

    x = R_earth * dlon * np.cos(ref_lat_rad)  # East
    y = R_earth * dlat  # North
    z = alt - ref_alt   # Up
    return np.array([x, y, z])


first_gt = next(iter(gt_data.values()))
ref_lat = first_gt["lat"]
ref_lon = first_gt["lon"]
ref_alt = first_gt["alt"]
print(f"Reference GPS: ({ref_lat}, {ref_lon}, {ref_alt})")

Reference GPS: (40.9067658, 29.15504999, 1.998676419)


## Adım 4: Training Görüntülerinden Transformation Matrix Hesapla

In [7]:
def _get_cam_from_world(image):
    cam_from_world = getattr(image, "cam_from_world", None)
    if cam_from_world is None:
        return None
    return cam_from_world() if callable(cam_from_world) else cam_from_world

def _rotation_matrix(rotation):
    if rotation is None:
        return None
    if hasattr(rotation, "matrix"):
        mat = rotation.matrix
        return mat() if callable(mat) else mat
    if hasattr(rotation, "as_matrix"):
        return rotation.as_matrix()
    return np.array(rotation)

def _translation_vector(transform):
    if transform is None:
        return None
    if hasattr(transform, "translation"):
        return transform.translation
    if hasattr(transform, "t"):
        return transform.t
    return None

def estimate_transform_matrix(model, gt_data, train_dir, ref_lat, ref_lon, ref_alt):
    """
    COLMAP frame -> GPS frame dönüşümünü train görüntülerinden hesaplar.
    Sim(3): p_gps = s * R * p_colmap + t
    """
    train_names = {p.name for p in train_dir.iterdir() if p.is_file()}

    colmap_positions = []
    gps_positions = []

    for _, image in model.images.items():
        img_name = Path(image.name).name
        if img_name not in train_names:
            continue
        if img_name not in gt_data:
            continue

        # COLMAP kamera merkezi (world): C = -R^T t
        if hasattr(image, "tvec") and hasattr(image, "qvec"):
            tvec = image.tvec
            qvec = image.qvec  # [w, x, y, z]
            rot = R.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])  # scipy: [x,y,z,w]
            R_cw = rot.as_matrix()
        else:
            cam_from_world = _get_cam_from_world(image)
            R_cw = _rotation_matrix(getattr(cam_from_world, "rotation", None))
            tvec = _translation_vector(cam_from_world)
        if R_cw is None or tvec is None:
            continue
        P_world = -R_cw.T @ tvec

        gt = gt_data[img_name]
        P_gps = gps_to_cartesian(gt["lat"], gt["lon"], gt["alt"], ref_lat, ref_lon, ref_alt)

        colmap_positions.append(P_world)
        gps_positions.append(P_gps)

    if len(colmap_positions) < 3:
        raise ValueError(
            f"Transformation için en az 3 eşleşme gerekli, bulunan: {len(colmap_positions)}"
        )

    colmap_positions = np.array(colmap_positions).T  # (3, N)
    gps_positions = np.array(gps_positions).T        # (3, N)

    print(f"Transformation için kullanılan train eşleşmesi: {colmap_positions.shape[1]}")

    # Umeyama alignment
    centroid_colmap = np.mean(colmap_positions, axis=1, keepdims=True)
    centroid_gps = np.mean(gps_positions, axis=1, keepdims=True)

    P_centered = colmap_positions - centroid_colmap
    Q_centered = gps_positions - centroid_gps

    H = P_centered @ Q_centered.T
    U, _, Vt = np.linalg.svd(H)
    R_opt = Vt.T @ U.T

    if np.linalg.det(R_opt) < 0:
        Vt[-1, :] *= -1
        R_opt = Vt.T @ U.T

    scale = np.trace(R_opt.T @ H) / np.trace(P_centered @ P_centered.T)
    t_opt = centroid_gps.flatten() - scale * R_opt @ centroid_colmap.flatten()

    # Kalibrasyon hatası
    P_transformed = scale * R_opt @ colmap_positions + t_opt.reshape(3, 1)
    errors = np.linalg.norm(P_transformed - gps_positions, axis=0)

    print(f"✓ Scale: {scale:.6f}")
    print(f"✓ Train alignment mean error: {np.mean(errors):.3f} m")
    print(f"✓ Train alignment median error: {np.median(errors):.3f} m")

    return scale, R_opt, t_opt


model = pycolmap.Reconstruction(sfm_model)
print(f"COLMAP model: {len(model.images)} image, {len(model.points3D)} point")

scale, R_transform, t_transform = estimate_transform_matrix(
    model=model,
    gt_data=gt_data,
    train_dir=train_images_dir,
    ref_lat=ref_lat,
    ref_lon=ref_lon,
    ref_alt=ref_alt,
)

COLMAP model: 106 image, 25020 point
Transformation için kullanılan train eşleşmesi: 106
✓ Scale: 1.386185
✓ Train alignment mean error: 31.846 m
✓ Train alignment median error: 31.886 m


## Adım 5: Test Görüntülerini Hazırla ve Lokalize Et

In [8]:
def extract_on_cpu(conf, image_root, image_list, feature_path):
    """CPU'da feature extraction (OOM önlemek için)"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    original = torch.cuda.is_available
    torch.cuda.is_available = lambda: False
    try:
        return extract_features.main(
            conf,
            image_root,
            image_list=image_list,
            feature_path=feature_path,
            overwrite=False,
        )
    finally:
        torch.cuda.is_available = original

# DB dosyaları (reconstruction'dan)
db_features = bundle_root / "features.h5"
db_global_features = bundle_root / "global-feats-netvlad.h5"

# Test images listesi
test_image_list = sorted([f for f in test_images_dir.iterdir() if f.is_file()])
print(f"Test seti: {len(test_image_list)} resim")

if len(test_image_list) == 0:
    print("UYARI: Test resim bulunamadı!")

Test seti: 27 resim


## Adım 6: Batch Lokalizasyon (Tüm Test Images)

In [9]:
def resolve_reference_ids(model, names):
    """Reference image isimlerini model image id'lerine dönüştür"""
    ref_ids = []
    missing = []
    for name in names:
        image = model.find_image_with_name(name)
        if image is None:
            normalized = name.replace('\\', '/').lstrip('./')
            candidates = [normalized]
            if normalized.startswith('images/'):
                candidates.append(normalized[len('images/'):])
            else:
                candidates.append(f"images/{Path(normalized).name}")
            for candidate in candidates:
                image = model.find_image_with_name(candidate)
                if image is not None:
                    break
        if image is None:
            missing.append(name)
            continue
        ref_ids.append(image.image_id)
    
    if missing:
        raise ValueError('SfM modelinde bulunamayan: ' + ', '.join(missing))
    return ref_ids

# Batch lokalizasyon sonuçları
localization_results = []
gt_poses = []

# Tüm test images üzerinde döngü
for idx, test_image_path in enumerate(test_image_list):
    print(f"\n[{idx+1}/{len(test_image_list)}] İşleniyor: {test_image_path.name}")
    
    # Query image'ı bundle'a kopyala
    query_dir = bundle_root / "query_batch"
    query_dir.mkdir(exist_ok=True)
    query_image_dst = query_dir / test_image_path.name
    shutil.copy2(test_image_path, query_image_dst)
    
    query_rel = f"query_batch/{query_image_dst.name}"
    loc_dir = results_dir / f"{test_image_path.stem}"
    loc_dir.mkdir(exist_ok=True)
    
    try:
        # Reference images (DB)
        references = sorted([f"images/{p.name}" for p in images_dir.iterdir() if p.is_file()])
        
        # Query features
        query_features = loc_dir / "query-features.h5"
        query_global_features = loc_dir / "query-global-feats-netvlad.h5"
        query_matches = loc_dir / "query-matches.h5"
        loc_pairs = loc_dir / "pairs-query-netvlad.txt"
        results_path = loc_dir / "query-pose.txt"
        
        # Feature extraction
        extract_on_cpu(
            retrieval_conf,
            bundle_root,
            [query_rel],
            query_global_features,
        )
        
        extract_on_cpu(
            feature_conf,
            bundle_root,
            [query_rel],
            query_features,
        )
        
        # Retrieval ve matching
        pairs_from_retrieval.main(
            descriptors=query_global_features,
            output=loc_pairs,
            num_matched=num_loc,
            query_list=[query_rel],
            db_list=references,
            db_descriptors=db_global_features,
        )
        
        match_features.main(
            matcher_conf,
            loc_pairs,
            features=query_features,
            features_ref=db_features,
            matches=query_matches,
            overwrite=True,
        )
        
        # Pose estimation
        retrieval_dict = parse_retrieval(loc_pairs)
        db_names = retrieval_dict[query_rel]
        ref_ids = resolve_reference_ids(model, db_names)
        
        conf = {
            "estimation": {"ransac": {"max_error": max_error}},
            "refinement": {"refine_focal_length": True, "refine_extra_params": True},
        }
        localizer = QueryLocalizer(model, conf)
        camera = pycolmap.infer_camera_from_image(bundle_root / query_rel)
        ret, log = pose_from_cluster(localizer, query_rel, camera, ref_ids, query_features, query_matches)
        
        if ret is None:
            print(f"  ✗ Pose estimation başarısız")
            localization_results.append({
                "image_name": test_image_path.name,
                "success": False,
                "inliers": 0,
            })
            continue
        
        print(f"  ✓ Lokalizasyon başarılı: {ret['num_inliers']} inliers")
        
        # COLMAP pose'unu GPS frame'ine dönüştür
        if scale is not None and R_transform is not None and t_transform is not None:
            # ret['cam_from_world'] = kamera pozisyonu COLMAP frame'inde
            # Dünya pozisyonu = -R^T @ t
            tvec = ret['cam_from_world'].translation
            qvec = ret['cam_from_world'].rotation.quat  # [w, x, y, z]
            
            rot = R.from_quat([qvec[1], qvec[2], qvec[3], qvec[0]])
            R_cw = rot.as_matrix()
            P_colmap = -R_cw.T @ tvec
            
            # Transform et
            P_gps_est = scale * R_transform @ P_colmap + t_transform
            
            # Rotation
            R_cw_gps = R_transform @ R_cw
            rot_gps = R.from_matrix(R_cw_gps)
            euler_gps = rot_gps.as_euler('xyz', degrees=True)
            
            localization_results.append({
                "image_name": test_image_path.name,
                "success": True,
                "inliers": ret['num_inliers'],
                "position_gps": P_gps_est,
                "euler_angles": euler_gps,
                "matches": len(db_names),
            })
        else:
            localization_results.append({
                "image_name": test_image_path.name,
                "success": True,
                "inliers": ret['num_inliers'],
                "position_colmap": -R_cw.T @ tvec,
            })
        
    except Exception as e:
        print(f"  ✗ Hata: {e}")
        localization_results.append({
            "image_name": test_image_path.name,
            "success": False,
            "error": str(e),
        })

print(f"\n✓ Batch lokalizasyon tamamlandı")

[2026/03/31 20:42:42 hloc INFO] Extracting local features with configuration:
{'model': {'name': 'netvlad'},
 'output': 'global-feats-netvlad',
 'preprocessing': {'resize_max': 1024}}



[1/27] İşleniyor: 000001.png


100%|██████████| 1/1 [00:05<00:00,  5.21s/it]
[2026/03/31 20:42:54 hloc INFO] Finished exporting features.
[2026/03/31 20:42:54 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-r1024',
 'preprocessing': {'grayscale': True, 'resize_max': 1024}}


Loaded SuperPoint model


c:\users\ilker\desktop\bitirme-project\hierarchical-localization\hloc\extractors\..\..\third_party\SuperGluePretrainedNetwork\models\superpoint.py:137: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this

...

  ✓ Lokalizasyon başarılı: 599 inliers

[27/27] İşleniyor: 001311.png


100%|██████████| 1/1 [00:04<00:00,  4.18s/it]
[2026/03/31 20:50:38 hloc INFO] Finished exporting features.
[2026/03/31 20:50:38 hloc INFO] Extracting local features with configuration:
{'model': {'max_keypoints': 4096, 'name': 'superpoint', 'nms_radius': 3},
 'output': 'feats-superpoint-n4096-r1024',
 'preprocessing': {'grayscale': True, 'resize_max': 1024}}


Loaded SuperPoint model


100%|██████████| 1/1 [00:04<00:00,  4.08s/it]
[2026/03/31 20:50:42 hloc INFO] Finished exporting features.
[2026/03/31 20:50:42 hloc INFO] Extracting image pairs from a retrieval database.
[2026/03/31 20:50:42 hloc INFO] Found 10 pairs.
[2026/03/31 20:50:42 hloc INFO] Matching local features with configuration:
{'model': {'features': 'superpoint', 'name': 'lightglue'},
 'output': 'matches-superpoint-lightglue'}
100%|██████████| 10/10 [00:04<00:00,  2.04it/s]
[2026/03/31 20:50:47 hloc INFO] Finished exporting matches.


  ✓ Lokalizasyon başarılı: 605 inliers

✓ Batch lokalizasyon tamamlandı


## Adım 7: Accuracy Raporu Oluştur

In [10]:
# Başarılı sonuçları filtrele
successful_results = [r for r in localization_results if r["success"]]
failed_results = [r for r in localization_results if not r["success"]]

position_errors = []
rotation_errors = []

print(f"\n{'='*60}")
print("BENCHMARK SONUÇLARI")
print(f"{'='*60}")
print(f"Toplam test images: {len(localization_results)}")
print(f"Başarılı: {len(successful_results)}")
print(f"Başarısız: {len(failed_results)}")
print(f"Success rate: {len(successful_results)/max(1,len(localization_results))*100:.1f}%")

for result in successful_results:
    img_name = result["image_name"]
    if img_name not in gt_data or "position_gps" not in result:
        continue

    gt = gt_data[img_name]
    P_gps_gt = gps_to_cartesian(gt["lat"], gt["lon"], gt["alt"], ref_lat, ref_lon, ref_alt)

    # Position error (metre)
    pos_error = np.linalg.norm(result["position_gps"] - P_gps_gt)
    position_errors.append(pos_error)

    # Rotation error (yaw/heading)
    heading_est = float(result["euler_angles"][2])
    heading_gt = float(gt["heading_deg"])
    rot_error = abs(heading_est - heading_gt)
    if rot_error > 180:
        rot_error = 360 - rot_error
    rotation_errors.append(rot_error)

if len(position_errors) == 0:
    print("\nGround truth eşleşen başarılı örnek bulunamadı.")
else:
    position_errors = np.array(position_errors)
    rotation_errors = np.array(rotation_errors)

    print(f"\n{'='*60}")
    print("POSITION ERROR (m)")
    print(f"{'='*60}")
    print(f"Mean:   {np.mean(position_errors):.3f}")
    print(f"Median: {np.median(position_errors):.3f}")
    print(f"Std:    {np.std(position_errors):.3f}")

    print(f"\n{'='*60}")
    print("ROTATION ERROR (deg)")
    print(f"{'='*60}")
    print(f"Mean:   {np.mean(rotation_errors):.2f}")
    print(f"Median: {np.median(rotation_errors):.2f}")
    print(f"Std:    {np.std(rotation_errors):.2f}")

    # Klasik Visual Localization benzeri eşik raporu
    print(f"\n{'='*60}")
    print("CLASSIC ACCURACY")
    print(f"{'='*60}")
    classic_thresholds = [
        (0.25, 2.0, "high-precision"),
        (0.50, 5.0, "medium"),
        (5.00, 10.0, "coarse"),
    ]
    for p_th, r_th, name in classic_thresholds:
        ok = (position_errors <= p_th) & (rotation_errors <= r_th)
        acc = 100.0 * np.mean(ok)
        print(f"{name:>14}: @ {p_th:.2f}m, {r_th:.1f}° -> {acc:5.1f}%")


BENCHMARK SONUÇLARI
Toplam test images: 27
Başarılı: 27
Başarısız: 0
Success rate: 100.0%

POSITION ERROR (m)
Mean:   36.188
Median: 37.467
Std:    22.009

ROTATION ERROR (deg)
Mean:   53.51
Median: 51.22
Std:    29.99

CLASSIC ACCURACY
high-precision: @ 0.25m, 2.0° ->   0.0%
        medium: @ 0.50m, 5.0° ->   0.0%
        coarse: @ 5.00m, 10.0° ->   0.0%


## Adım 8: Sonuçları Kaydet

In [ ]:
# Sonuçları JSON olarak kaydet
results_json = results_dir / "benchmark_summary.json"

summary = {
    "timestamp": datetime.now().isoformat(),
    "total_test_images": len(localization_results),
    "successful_localizations": len(successful_results),
    "failed_localizations": len(failed_results),
    "success_rate": len(successful_results) / len(localization_results) if len(localization_results) > 0 else 0.0,
    "bundle_root": str(bundle_root),
    "train_images_dir": str(train_images_dir),
    "test_images_dir": str(test_images_dir),
    "ground_truth_csv": str(ground_truth_csv),
    "parameters": {
        "num_retrieved": num_loc,
        "ransac_max_error": max_error,
    },
}

if len(position_errors) > 0:
    summary["position_accuracy"] = {
        "mean_m": float(np.mean(position_errors)),
        "median_m": float(np.median(position_errors)),
        "std_m": float(np.std(position_errors)),
    }

if len(rotation_errors) > 0:
    summary["rotation_accuracy"] = {
        "mean_deg": float(np.mean(rotation_errors)),
        "median_deg": float(np.median(rotation_errors)),
        "std_deg": float(np.std(rotation_errors)),
    }

if len(position_errors) > 0 and len(rotation_errors) > 0:
    classic_thresholds = [
        (0.25, 2.0, "high_precision"),
        (0.50, 5.0, "medium"),
        (5.00, 10.0, "coarse"),
    ]
    classic = {}
    pe = np.array(position_errors)
    re = np.array(rotation_errors)
    for p_th, r_th, name in classic_thresholds:
        classic[name] = float(np.mean((pe <= p_th) & (re <= r_th)))
    summary["classic_accuracy"] = classic

with open(results_json, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Sonuçlar kaydedildi: {results_json}")

# Detaylı CSV
details_csv = results_dir / "localization_details.csv"
df = pd.DataFrame(localization_results)
df.to_csv(details_csv, index=False)
print(f"✓ Detaylar kaydedildi: {details_csv}")